[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/hausa-asr-transformer/blob/main/capstone_demo.ipynb)

In [ ]:
# Run this cell if executing in Google Colab to install dependencies
import sys
if 'google.colab' in sys.modules:
    !pip install transformers datasets torch torchaudio librosa evaluate jiwer accelerate

## 1. Architecture Overview

## 2. Audio Exploratory Data Analysis

In [ ]:
import io

import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from datasets import load_dataset, Audio as HFAudio
from IPython.display import Audio, display

SAMPLING_RATE = 16_000

# Using google/fleurs (ha_ng) instead of mozilla-foundation/common_voice_11_0:
# Mozilla moved Common Voice off the HF Hub to "Mozilla Data Collective" in Oct 2025.
ds = load_dataset("google/fleurs", "ha_ng", split="train")
ds = ds.cast_column("audio", HFAudio(sampling_rate=SAMPLING_RATE, decode=False))

samples = [ds[i] for i in range(3)]

fig, axes = plt.subplots(3, 2, figsize=(12, 9))

for i, sample in enumerate(samples):
    audio_array, sr = sf.read(io.BytesIO(sample["audio"]["bytes"]))

    print(f"Sample {i + 1}: {sample['raw_transcription']}")
    display(Audio(audio_array, rate=sr))

    librosa.display.waveshow(audio_array, sr=sr, ax=axes[i, 0])
    axes[i, 0].set_title(f"Sample {i + 1} — Waveform")

    mel_spec = librosa.feature.melspectrogram(y=audio_array, sr=sr, n_mels=80)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    librosa.display.specshow(mel_spec_db, sr=sr, x_axis="time", y_axis="mel", ax=axes[i, 1])
    axes[i, 1].set_title(f"Sample {i + 1} — Mel-Spectrogram")

plt.tight_layout()
plt.show()

## 3. Data Pipeline Verification

## 4. Training Telemetry

In [ ]:
import glob
import json
import os

import matplotlib.pyplot as plt

# Requires train.py to have been run at least once, writing checkpoints under OUTPUT_DIR.
OUTPUT_DIR = "./whisper-small-ha"

checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[-1]))
state_path = f"{checkpoints[-1]}/trainer_state.json" if checkpoints else f"{OUTPUT_DIR}/trainer_state.json"

if not os.path.exists(state_path):
    print(f"No training checkpoints found at '{OUTPUT_DIR}'. Run train.py (or Section 4's training cell) first.")
else:
    with open(state_path) as f:
        log_history = json.load(f)["log_history"]

    train_steps = [e["step"] for e in log_history if "loss" in e]
    train_loss = [e["loss"] for e in log_history if "loss" in e]

    eval_steps = [e["step"] for e in log_history if "eval_wer" in e]
    eval_wer = [e["eval_wer"] for e in log_history if "eval_wer" in e]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(train_steps, train_loss, marker="o")
    axes[0].set_title("Training Loss")
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Loss")

    axes[1].plot(eval_steps, eval_wer, marker="o", color="darkorange")
    axes[1].set_title("Word Error Rate (WER)")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("WER (%)")

    plt.tight_layout()
    plt.show()

## 5. Interactive Inference

In [ ]:
import io
import os

import jiwer
import pandas as pd
import soundfile as sf
import torch
from datasets import load_dataset, Audio as HFAudio
from transformers import WhisperForConditionalGeneration, WhisperProcessor

# Requires train.py to have been run at least once, saving the fine-tuned model to MODEL_DIR.
MODEL_DIR = "./whisper-small-ha"
SAMPLING_RATE = 16_000

if not os.path.exists(MODEL_DIR):
    print(f"No fine-tuned model found at '{MODEL_DIR}'. Run train.py (or Section 4's training cell) first.")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = WhisperProcessor.from_pretrained(MODEL_DIR)
    model = WhisperForConditionalGeneration.from_pretrained(MODEL_DIR).to(device)

    test_ds = load_dataset("google/fleurs", "ha_ng", split="test")
    test_ds = test_ds.cast_column("audio", HFAudio(sampling_rate=SAMPLING_RATE, decode=False))
    samples = [test_ds[i] for i in range(5)]

    rows = []
    for sample in samples:
        audio_array, sr = sf.read(io.BytesIO(sample["audio"]["bytes"]))
        input_features = processor.feature_extractor(
            audio_array, sampling_rate=sr, return_tensors="pt"
        ).input_features.to(device)

        predicted_ids = model.generate(input_features, language="hausa", task="transcribe")
        prediction = processor.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()

        ground_truth = sample["raw_transcription"]
        wer = jiwer.wer(ground_truth, prediction) * 100

        rows.append({
            "Sample ID": sample["id"],
            "Ground Truth Hausa": ground_truth,
            "Model Output": prediction,
            "WER Score": round(wer, 2),
        })

    results_df = pd.DataFrame(rows)
    results_df